In [0]:
CREATE OR REPLACE TABLE workspace.retail_project.silver_online_retail
COMMENT 'Cleaned transaction-line data; source rows retained, missing values preserved.'
AS
SELECT 
    TRIM(InvoiceNo) AS InvoiceNo,
    TRIM(StockCode) AS StockCode,
    Description,
    TRY_CAST(TRIM(Quantity) AS INT) AS Quantity,
    TRY_TO_TIMESTAMP(TRIM(InvoiceDate), 'M/d/yyyy H:mm') AS InvoiceDate,
    TRY_CAST(TRIM(UnitPrice) AS DECIMAL(18, 4)) AS UnitPrice,
    TRIM(CustomerID) AS CustomerID,
    Country,
    CASE
        WHEN UPPER(TRIM(InvoiceNo)) LIKE 'C%' THEN TRUE
        ELSE FALSE 
    END AS is_cancellation,
    CASE
        WHEN TRY_CAST(TRIM(Quantity) AS INT) < 0 THEN TRUE
        ELSE FALSE 
    END AS is_negative_quantity
FROM workspace.retail_project.bronze_online_retail_raw;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*)
FROM workspace.retail_project.silver_online_retail

count(1)
541909


In [0]:
SELECT
    SUM(CASE
        WHEN is_cancellation = TRUE THEN 1
        ELSE 0
    END) AS cancellation_rows,

    SUM(CASE
        WHEN is_negative_quantity = TRUE THEN 1
        ELSE 0
    END) AS negative_quantity_rows,

    SUM(CASE
        WHEN is_cancellation = TRUE
         AND is_negative_quantity = TRUE
        THEN 1
        ELSE 0
    END) AS both_flags_true

FROM workspace.retail_project.silver_online_retail;

cancellation_rows,negative_quantity_rows,both_flags_true
9288,10624,9288


In [0]:
SELECT
    SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END)
        AS null_quantity_rows,

    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END)
        AS null_unitprice_rows,

    SUM(CASE WHEN InvoiceDate IS NULL THEN 1 ELSE 0 END)
        AS null_invoicedate_rows

FROM workspace.retail_project.silver_online_retail;

null_quantity_rows,null_unitprice_rows,null_invoicedate_rows
0,0,0


In [0]:
WITH silver_counts AS (
    SELECT
        SUM(CASE WHEN CustomerID  IS NULL THEN 1 ELSE 0 END) AS silver_null_customerid,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS silver_null_description
    FROM workspace.retail_project.silver_online_retail
),

bronze_counts AS (
    SELECT
        SUM(CASE WHEN CustomerID  IS NULL THEN 1 ELSE 0 END) AS bronze_null_customerid,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS bronze_null_description
    FROM workspace.retail_project.bronze_online_retail_raw
)

SELECT
    b.bronze_null_customerid,
    s.silver_null_customerid,
    s.silver_null_customerid - b.bronze_null_customerid AS customerid_diff,
    CASE
        WHEN s.silver_null_customerid = b.bronze_null_customerid THEN 'PASS'
        ELSE 'CHECK'
    END AS customerid_validation,
    b.bronze_null_description,
    s.silver_null_description,
    s.silver_null_description - b.bronze_null_description AS description_diff,
    CASE
        WHEN s.silver_null_description = b.bronze_null_description THEN 'PASS'
        ELSE 'CHECK'
    END AS description_validation

FROM bronze_counts b
CROSS JOIN silver_counts s;

bronze_null_customerid,silver_null_customerid,customerid_diff,customerid_validation,bronze_null_description,silver_null_description,description_diff,description_validation
135080,135080,0,PASS,1454,1454,0,PASS
